# 03 · Train the proposed model (full `PENet`)
Trains the vessel-topology-aware multi-task network with all components enabled, selects decision thresholds on the **validation** set, and saves the checkpoint + thresholds used by notebooks 04–07.

Typical Kaggle settings: GPU T4/P100, `cfg.vol_shape=(64,128,128)`, `bs=2`, `accum=4`. Training time scales with the number of studies; do a smoke test with `SUBSET=64` first.

In [ ]:
%run ./00_common.ipynb

In [ ]:
sp = load_splits()
SUBSET = None                                   # e.g. 64 for a smoke test
tr, va, te = sp["train"][:SUBSET], sp["val"][:SUBSET], sp["test"][:SUBSET]
fl = Flags(); model = PENet(cfg, fl).to(DEVICE)
print(f"{fl.name}\nparams: {count_params(model):.2f} M")

# --- shape / loss sanity check on one batch before the long run
b = to_dev(next(iter(make_loader(tr[:cfg.bs * 2], cfg, True))))
out = model(b); loss, parts = compute_loss(out, b, cfg)
print({k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)}); print("loss", float(loss), parts)

In [ ]:
set_seed(cfg.seed)
model, hist, best = train_model(model, tr, va, cfg, epochs=cfg.epochs, tag="penet_full")
import pandas as pd; H = pd.DataFrame(hist); H.to_csv(f"{cfg.out_dir}/penet_full_history.csv", index=False); print("best val AUC:", best)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for c in [c for c in H.columns if c.startswith("loss_")]: ax[0].plot(H[c], label=c[5:])
ax[0].set_title("training loss terms"); ax[0].legend(fontsize=7)
ax[1].plot(H.val_auc, label="val AUC"); ax[1].plot(H.val_ap, label="val AP"); ax[1].legend(); ax[1].set_title("detection")
ax[2].plot(H.val_slice_f1); ax[2].set_title("val slice-level F1 (localisation)"); plt.tight_layout(); plt.show()

### Threshold selection on validation set → saved for all later notebooks

In [ ]:
Rv = predict(model, make_loader(va, cfg, False))
thr_slice, f1 = best_slice_thr(Rv["slice_prob"], Rv["slice_lab"], Rv["slice_valid"])
from sklearn.metrics import roc_curve
fpr, tpr, th = roc_curve(Rv["y"][:, 0], Rv["det_prob"]); thr_det = float(th[np.argmax(tpr - fpr)])      # Youden J
json.dump(dict(thr_det=thr_det, thr_slice=thr_slice), open(f"{cfg.ckpt_dir}/thresholds.json", "w"))
print(f"thr_det={thr_det:.3f}  thr_slice={thr_slice:.2f} (val slice F1={f1:.3f})")
Rt = predict(model, make_loader(te, cfg, False)); print("TEST", det_metrics(Rt))